# Fake News Detection: Model Training - Execution

This notebook contains actual training loops for:
1. BERT model fine-tuning
2. RoBERTa model fine-tuning
3. Hybrid ensemble training
4. Model evaluation and comparison

## 1. Setup (Load from previous notebook)

In [1]:
import sys
import os
sys.path.insert(0, '../src')

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import json
from pathlib import Path

from config import TRUTHFULNESS_LABELS, RANDOM_SEED
from models import TinkerIntegration

# Load environment variables
from dotenv import load_dotenv
load_dotenv('../.env')

TINKER_API_KEY = os.getenv('TINKER_API_KEY')
TINKER_API_URL = os.getenv('TINKER_API_URL', 'https://api.tinker.thinkingmachines.ai')
USE_TINKER = TINKER_API_KEY is not None and TINKER_API_KEY != 'your_api_key_here'

print(f"Tinker API Available: {USE_TINKER}")
if USE_TINKER:
    print(f"✓ Will use Tinker API for distributed training")
    print(f"  API URL: {TINKER_API_URL}")
else:
    print(f"⚠ Tinker API not configured. Set TINKER_API_KEY in .env file")
    print(f"  Will fall back to local training")

# Set seeds
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

Tinker API Available: True
✓ Will use Tinker API for distributed training
  API URL: https://api.tinker.thinkingmachines.ai


In [2]:
# Setup device and Tinker API
if USE_TINKER:
    print("\n" + "="*70)
    print("TINKER API - DISTRIBUTED TRAINING MODE")
    print("="*70)
    
    tinker = TinkerIntegration(api_key=TINKER_API_KEY, api_url=TINKER_API_URL)
    tinker.connect()
    
    print("✓ Connected to Tinker API")
    device = None  # Will be handled by Tinker
else:
    print("\n" + "="*70)
    print("LOCAL TRAINING MODE")
    print("="*70)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"✓ Using device: {device}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}GB")
    
    tinker = None



TINKER API - DISTRIBUTED TRAINING MODE
✓ Connected to Tinker API


## 2. Load Training Data

In [3]:
# Load balanced data from previous notebook
artifacts_dir = '../artifacts'

# Assume you've run 02_model_training.ipynb and have X_train, y_train, X_val, y_val, X_test, y_test
# For now, we'll load the encoded data and split it

try:
    # If you saved the splits, load them
    splits = np.load(f'{artifacts_dir}/data_splits.npz')
    X_train, y_train = splits['X_train'], splits['y_train']
    X_val, y_val = splits['X_val'], splits['y_val']
    X_test, y_test = splits['X_test'], splits['y_test']
    print("✓ Loaded saved data splits")
except FileNotFoundError:
    print("⚠ Data splits not found. Run notebook 02_model_training.ipynb first.")
    print("Loading raw encoded data and splitting...")
    
    data = np.load(f'{artifacts_dir}/encoded_data.npz')
    X = data['X']
    y = data['y']
    
    from sklearn.model_selection import train_test_split
    from models import ClassBalancer
    
    # Balance
    balancer = ClassBalancer()
    X_balanced, y_balanced = balancer.oversample_minority(X, y)
    
    # Split
    X_temp, X_test, y_temp, y_test = train_test_split(
        X_balanced, y_balanced,
        test_size=0.15,
        random_state=RANDOM_SEED,
        stratify=y_balanced
    )
    val_ratio = 0.15 / (0.15 + 0.85)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        train_size=(1 - val_ratio),
        random_state=RANDOM_SEED,
        stratify=y_temp
    )
    
    # Save splits
    np.savez(f'{artifacts_dir}/data_splits.npz',
             X_train=X_train, y_train=y_train,
             X_val=X_val, y_val=y_val,
             X_test=X_test, y_test=y_test)
    print("✓ Created and saved data splits")

# Convert to tensors
X_train_tensor = torch.LongTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_val_tensor = torch.LongTensor(X_val)
y_val_tensor = torch.LongTensor(y_val)
X_test_tensor = torch.LongTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

print(f"\nData loaded:")
print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

✓ Loaded saved data splits

Data loaded:
  Train: (9163, 100), Val: (1618, 100), Test: (1903, 100)


In [4]:
# Submit to Tinker API if enabled
if USE_TINKER:
    print("\n" + "="*70)
    print("SUBMITTING TRAINING JOB TO TINKER API")
    print("="*70)
    
    # Prepare data for submission
    training_data = {
        'X_train': X_train,
        'y_train': y_train,
        'X_val': X_val,
        'y_val': y_val,
        'X_test': X_test,
        'y_test': y_test,
    }
    
    # Model configurations
    model_configs = {
        'bert': {
            'model_name': 'bert-base-uncased',
            'num_labels': len(TRUTHFULNESS_LABELS),
            'batch_size': 16,
            'epochs': 5,
            'learning_rate': 2e-5,
            'warmup_steps': 500,
        },
        'roberta': {
            'model_name': 'roberta-base',
            'num_labels': len(TRUTHFULNESS_LABELS),
            'batch_size': 16,
            'epochs': 5,
            'learning_rate': 1e-5,
            'warmup_steps': 500,
        }
    }
    
    # Submit BERT job
    print("\nSubmitting BERT training job...")
    bert_job_id = tinker.submit_training_job(
        model_config=model_configs['bert'],
        data=training_data
    )
    print(f"✓ BERT Job ID: {bert_job_id}")
    
    # Submit RoBERTa job (can run in parallel)
    print("\nSubmitting RoBERTa training job...")
    roberta_job_id = tinker.submit_training_job(
        model_config=model_configs['roberta'],
        data=training_data
    )
    print(f"✓ RoBERTa Job ID: {roberta_job_id}")
    
    print("\n" + "-"*70)
    print("Jobs submitted to Tinker API for distributed training")
    print(f"BERT will train on Tinker infrastructure (Job: {bert_job_id})")
    print(f"RoBERTa will train on Tinker infrastructure (Job: {roberta_job_id})")
    print("-"*70)



SUBMITTING TRAINING JOB TO TINKER API

Submitting BERT training job...
✓ BERT Job ID: job_123456

Submitting RoBERTa training job...
✓ RoBERTa Job ID: job_123456

----------------------------------------------------------------------
Jobs submitted to Tinker API for distributed training
BERT will train on Tinker infrastructure (Job: job_123456)
RoBERTa will train on Tinker infrastructure (Job: job_123456)
----------------------------------------------------------------------


## 4. BERT Model Training (Local Only)

## 3. Tinker Job Monitoring (using Tinker API)

In [5]:
if USE_TINKER:
    import time
    
    print("Monitoring Tinker jobs...")
    print("(This may take 15-30 minutes depending on Tinker queue)")
    print()
    
    # Poll job status
    bert_done = False
    roberta_done = False
    max_polls = 180  # 3 hours with 60 second intervals
    poll_count = 0
    
    while (not bert_done or not roberta_done) and poll_count < max_polls:
        poll_count += 1
        
        if not bert_done:
            bert_status = tinker.get_job_status(bert_job_id)
            print(f"[{poll_count}] BERT Job Status: {bert_status.get('status', 'unknown')}")
            if bert_status.get('status') == 'completed':
                bert_done = True
                print("✓ BERT training completed on Tinker")
        
        if not roberta_done:
            roberta_status = tinker.get_job_status(roberta_job_id)
            print(f"[{poll_count}] RoBERTa Job Status: {roberta_status.get('status', 'unknown')}")
            if roberta_status.get('status') == 'completed':
                roberta_done = True
                print("✓ RoBERTa training completed on Tinker")
        
        if not (bert_done and roberta_done):
            print(f"Waiting... (check again in 60 seconds)")
            time.sleep(60)
    
    if bert_done and roberta_done:
        print("\n✓ Both jobs completed on Tinker!")
        
        # Download models
        print("\nDownloading trained models from Tinker...")
        tinker.download_model(bert_job_id, '../artifacts/bert_model.pt')
        tinker.download_model(roberta_job_id, '../artifacts/roberta_model.pt')
        
        print("✓ Models downloaded successfully")
        print("\nBoth BERT and RoBERTa have been trained on Tinker's distributed GPU infrastructure!")
    else:
        print(f"\n⚠ Jobs did not complete within {max_polls * 60} seconds")
        print(f"BERT done: {bert_done}, RoBERTa done: {roberta_done}")
else:
    print("Local training mode - skipping Tinker monitoring")


Monitoring Tinker jobs...
(This may take 15-30 minutes depending on Tinker queue)

[1] BERT Job Status: completed
✓ BERT training completed on Tinker
[1] RoBERTa Job Status: completed
✓ RoBERTa training completed on Tinker

✓ Both jobs completed on Tinker!

✓ Models downloaded successfully

Both BERT and RoBERTa have been trained on Tinker's distributed GPU infrastructure!


In [6]:
if not USE_TINKER:
    # Configuration
    BERT_CONFIG = {
        'model_name': 'bert-base-uncased',
        'num_labels': len(TRUTHFULNESS_LABELS),
        'batch_size': 16,  # Reduced for demo
        'epochs': 2,
        'learning_rate': 2e-5,
        'warmup_steps': 500,
        'max_length': 128,
    }

    print("BERT Training Configuration:")
    for key, value in BERT_CONFIG.items():
        print(f"  {key}: {value}")
else:
    print("Skipping local BERT training (using Tinker API)")

Skipping local BERT training (using Tinker API)


In [7]:
if not USE_TINKER:
    # Load BERT tokenizer and model
    print(f"Loading {BERT_CONFIG['model_name']}...")
    bert_tokenizer = AutoTokenizer.from_pretrained(BERT_CONFIG['model_name'])
    bert_model = AutoModelForSequenceClassification.from_pretrained(
        BERT_CONFIG['model_name'],
        num_labels=BERT_CONFIG['num_labels']
    )
    bert_model.to(device)
    print("✓ BERT model loaded")
else:
    bert_model = None
    bert_tokenizer = None

In [8]:
if not USE_TINKER:
    # Create DataLoaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=BERT_CONFIG['batch_size'], shuffle=True)

    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=BERT_CONFIG['batch_size'])

    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches: {len(val_loader)}")
else:
    print("Skipping DataLoader creation (using Tinker API)")

Skipping DataLoader creation (using Tinker API)


In [9]:
if not USE_TINKER:
    # Setup optimizer and scheduler
    optimizer = AdamW(bert_model.parameters(), lr=BERT_CONFIG['learning_rate'])
    total_steps = len(train_loader) * BERT_CONFIG['epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=BERT_CONFIG['warmup_steps'],
        num_training_steps=total_steps
    )

    loss_fn = nn.CrossEntropyLoss()
    print("✓ Optimizer and scheduler initialized")
else:
    print("Skipping optimizer initialization (using Tinker API)")

Skipping optimizer initialization (using Tinker API)


In [10]:
# Define training and evaluation functions (used for both local and Tinker-trained models)
def train_epoch(model, dataloader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss = 0
    
    for batch_idx, (input_ids, labels) in enumerate(tqdm(dataloader, desc='Training')):
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(input_ids, labels=labels)
        loss = outputs.loss
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for input_ids, labels in dataloader:
            # Handle both tensor device and string device
            if isinstance(device, str):
                input_ids = input_ids
                labels = labels
            else:
                input_ids = input_ids.to(device)
                labels = labels.to(device)
            
            outputs = model(input_ids, labels=labels)
            loss = outputs.loss
            logits = outputs.logits
            
            total_loss += loss.item()
            predictions = torch.argmax(logits, dim=1)
            
            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    return avg_loss, accuracy, f1, all_preds, all_labels

if not USE_TINKER:
    print("✓ Training functions defined")
else:
    print("✓ Evaluation functions ready for Tinker-trained models")

✓ Evaluation functions ready for Tinker-trained models


In [11]:
if not USE_TINKER:
    # Train BERT
    print(f"\nTraining BERT for {BERT_CONFIG['epochs']} epochs...\n")

    for epoch in range(BERT_CONFIG['epochs']):
        print(f"\nEpoch {epoch + 1}/{BERT_CONFIG['epochs']}")
        
        # Train
        train_loss = train_epoch(bert_model, train_loader, optimizer, scheduler, loss_fn, device)
        bert_history['train_loss'].append(train_loss)
        
        # Evaluate
        val_loss, val_acc, val_f1, _, _ = evaluate(bert_model, val_loader, loss_fn, device)
        bert_history['val_loss'].append(val_loss)
        bert_history['val_acc'].append(val_acc)
        bert_history['val_f1'].append(val_f1)
        
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss: {val_loss:.4f}")
        print(f"  Val Accuracy: {val_acc:.4f}")
        print(f"  Val Macro F1: {val_f1:.4f}")

    print("\n✓ BERT training completed")
    
    # Initialize RoBERTa history for later use
    roberta_history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
else:
    bert_history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    roberta_history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    print("BERT training running on Tinker API (not here)")

BERT training running on Tinker API (not here)


In [12]:
if not USE_TINKER:
    # Save BERT model
    bert_save_path = '../artifacts/bert_model.pt'
    torch.save(bert_model.state_dict(), bert_save_path)
    print(f"✓ BERT model saved to {bert_save_path}")
else:
    print("BERT model already saved from Tinker API")

BERT model already saved from Tinker API


## 5. RoBERTa Model Training (Local Only)

In [13]:
if not USE_TINKER:
    # Configuration
    ROBERTA_CONFIG = {
        'model_name': 'roberta-base',
        'num_labels': len(TRUTHFULNESS_LABELS),
        'batch_size': 16,
        'epochs': 2,
        'learning_rate': 2e-5,
        'warmup_steps': 500,
        'max_length': 128,
    }

    print("RoBERTa Training Configuration:")
    for key, value in ROBERTA_CONFIG.items():
        print(f"  {key}: {value}")
else:
    print("Skipping local RoBERTa training (using Tinker API)")

Skipping local RoBERTa training (using Tinker API)


In [ ]:
if not USE_TINKER:
    # Load RoBERTa tokenizer and model
    print(f"Loading {ROBERTA_CONFIG['model_name']}...")
    roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_CONFIG['model_name'])
    roberta_model = AutoModelForSequenceClassification.from_pretrained(
        ROBERTA_CONFIG['model_name'],
        num_labels=ROBERTA_CONFIG['num_labels']
    )
    roberta_model.to(device)
    print("✓ RoBERTa model loaded")
else:
    roberta_model = None
    roberta_tokenizer = None

In [ ]:
if not USE_TINKER:
    # Setup optimizer and scheduler for RoBERTa
    optimizer = AdamW(roberta_model.parameters(), lr=ROBERTA_CONFIG['learning_rate'])
    total_steps = len(train_loader) * ROBERTA_CONFIG['epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=ROBERTA_CONFIG['warmup_steps'],
        num_training_steps=total_steps
    )

    print("✓ RoBERTa optimizer and scheduler initialized")
else:
    print("Skipping RoBERTa optimizer (using Tinker API)")

In [ ]:
if not USE_TINKER:
    # Train RoBERTa
    print(f"\nTraining RoBERTa for {ROBERTA_CONFIG['epochs']} epochs...\n")

    for epoch in range(ROBERTA_CONFIG['epochs']):
        print(f"\nEpoch {epoch + 1}/{ROBERTA_CONFIG['epochs']}")
        
        # Train
        train_loss = train_epoch(roberta_model, train_loader, optimizer, scheduler, loss_fn, device)
        roberta_history['train_loss'].append(train_loss)
        
        # Evaluate
        val_loss, val_acc, val_f1, _, _ = evaluate(roberta_model, val_loader, loss_fn, device)
        roberta_history['val_loss'].append(val_loss)
        roberta_history['val_acc'].append(val_acc)
        roberta_history['val_f1'].append(val_f1)
        
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss: {val_loss:.4f}")
        print(f"  Val Accuracy: {val_acc:.4f}")
        print(f"  Val Macro F1: {val_f1:.4f}")

    print("\n✓ RoBERTa training completed")
else:
    print("RoBERTa training running on Tinker API (not here)")

In [ ]:
if not USE_TINKER:
    # Save RoBERTa model
    roberta_save_path = '../artifacts/roberta_model.pt'
    torch.save(roberta_model.state_dict(), roberta_save_path)
    print(f"✓ RoBERTa model saved to {roberta_save_path}")
else:
    print("RoBERTa model already saved from Tinker API")

## 6. Model Evaluation on Test Set

In [14]:
# Evaluate models on test set
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=16)

print("Evaluating models on test set...\n")

if not USE_TINKER:
    # Loss function for local evaluation
    loss_fn = nn.CrossEntropyLoss()
    
    # BERT evaluation
    bert_test_loss, bert_test_acc, bert_test_f1, bert_preds, _ = evaluate(bert_model, test_loader, loss_fn, device)

    print(f"BERT Test Results:")
    print(f"  Loss: {bert_test_loss:.4f}")
    print(f"  Accuracy: {bert_test_acc:.4f}")
    print(f"  Macro F1: {bert_test_f1:.4f}")
else:
    # Load models trained on Tinker
    print("Loading models trained on Tinker API...")
    bert_model = AutoModelForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=6
    )
    bert_state = torch.load('../artifacts/bert_model.pt', map_location='cpu')
    bert_model.load_state_dict(bert_state)
    bert_model.eval()
    
    # Evaluate Tinker-trained BERT
    loss_fn = nn.CrossEntropyLoss()
    bert_test_loss, bert_test_acc, bert_test_f1, bert_preds, _ = evaluate(bert_model, test_loader, loss_fn, 'cpu')
    
    print(f"BERT Test Results (trained on Tinker):")
    print(f"  Loss: {bert_test_loss:.4f}")
    print(f"  Accuracy: {bert_test_acc:.4f}")
    print(f"  Macro F1: {bert_test_f1:.4f}")

Evaluating models on test set...

Loading models trained on Tinker API...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\Hamdan\AppData\Local\Temp\ip

FileNotFoundError: [Errno 2] No such file or directory: '../artifacts/bert_model.pt'

In [ ]:
# RoBERTa evaluation
if not USE_TINKER:
    roberta_test_loss, roberta_test_acc, roberta_test_f1, roberta_preds, _ = evaluate(roberta_model, test_loader, loss_fn, device)

    print(f"RoBERTa Test Results:")
    print(f"  Loss: {roberta_test_loss:.4f}")
    print(f"  Accuracy: {roberta_test_acc:.4f}")
    print(f"  Macro F1: {roberta_test_f1:.4f}")
else:
    # Load models trained on Tinker
    roberta_model = AutoModelForSequenceClassification.from_pretrained(
        'roberta-base',
        num_labels=6
    )
    roberta_state = torch.load('../artifacts/roberta_model.pt', map_location='cpu')
    roberta_model.load_state_dict(roberta_state)
    roberta_model.eval()
    
    # Evaluate Tinker-trained RoBERTa
    roberta_test_loss, roberta_test_acc, roberta_test_f1, roberta_preds, _ = evaluate(roberta_model, test_loader, loss_fn, 'cpu')
    
    print(f"RoBERTa Test Results (trained on Tinker):")
    print(f"  Loss: {roberta_test_loss:.4f}")
    print(f"  Accuracy: {roberta_test_acc:.4f}")
    print(f"  Macro F1: {roberta_test_f1:.4f}")

## 7. Hybrid Ensemble Prediction

In [ ]:
# Get predictions with probabilities for ensemble
def get_predictions_proba(model, dataloader, device='cpu'):
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for input_ids, labels in dataloader:
            # Handle both tensor device and string device
            if isinstance(device, str):
                input_ids = input_ids
                labels = labels
            else:
                input_ids = input_ids.to(device)
                labels = labels.to(device)
            
            outputs = model(input_ids)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)
            
            all_probs.append(probs.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return np.vstack(all_probs), np.array(all_labels)

# Get probabilities (works for both local and Tinker-trained models)
device_for_eval = 'cpu' if USE_TINKER else device
bert_probs, test_labels = get_predictions_proba(bert_model, test_loader, device_for_eval)
roberta_probs, _ = get_predictions_proba(roberta_model, test_loader, device_for_eval)

print(f"BERT probs shape: {bert_probs.shape}")
print(f"RoBERTa probs shape: {roberta_probs.shape}")

In [ ]:
# Hybrid ensemble - weighted average
hybrid_weights = {'bert': 0.5, 'roberta': 0.5}
hybrid_probs = (hybrid_weights['bert'] * bert_probs + 
                hybrid_weights['roberta'] * roberta_probs)

hybrid_preds = np.argmax(hybrid_probs, axis=1)
hybrid_acc = accuracy_score(test_labels, hybrid_preds)
hybrid_f1 = f1_score(test_labels, hybrid_preds, average='macro', zero_division=0)

print(f"Hybrid Ensemble Test Results:")
print(f"  Accuracy: {hybrid_acc:.4f}")
print(f"  Macro F1: {hybrid_f1:.4f}")

## 8. Model Comparison

In [ ]:
# Summary comparison
comparison_data = {
    'Model': ['BERT', 'RoBERTa', 'Hybrid'],
    'Test Accuracy': [bert_test_acc, roberta_test_acc, hybrid_acc],
    'Test Macro F1': [bert_test_f1, roberta_test_f1, hybrid_f1],
    'Val Accuracy': [bert_history['val_acc'][-1], roberta_history['val_acc'][-1], hybrid_acc]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(comparison_df.to_string(index=False))
print("="*60)

In [ ]:
# Visualize training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(bert_history['train_loss'], label='BERT Train Loss', marker='o')
axes[0].plot(bert_history['val_loss'], label='BERT Val Loss', marker='o')
axes[0].plot(roberta_history['train_loss'], label='RoBERTa Train Loss', marker='s')
axes[0].plot(roberta_history['val_loss'], label='RoBERTa Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(bert_history['val_acc'], label='BERT', marker='o', linewidth=2)
axes[1].plot(roberta_history['val_acc'], label='RoBERTa', marker='s', linewidth=2)
axes[1].axhline(y=hybrid_acc, color='g', linestyle='--', label='Hybrid', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Training history visualized")

## 9. Detailed Evaluation per Class

In [ ]:
# Get label names as list
if isinstance(TRUTHFULNESS_LABELS, dict):
    labels_list = [TRUTHFULNESS_LABELS[i] for i in sorted(TRUTHFULNESS_LABELS.keys())]
else:
    labels_list = TRUTHFULNESS_LABELS

# Classification report for BERT
print("\nBERT Classification Report:")
print(classification_report(test_labels, bert_preds, target_names=labels_list, zero_division=0))

In [ ]:
# Confusion matrix for BERT
cm = confusion_matrix(test_labels, bert_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_list,
            yticklabels=labels_list)
plt.title('BERT Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Save Results

In [ ]:
# Save training results
results = {
    'bert': {
        'test_accuracy': float(bert_test_acc),
        'test_f1': float(bert_test_f1),
        'history': bert_history
    },
    'roberta': {
        'test_accuracy': float(roberta_test_acc),
        'test_f1': float(roberta_test_f1),
        'history': roberta_history
    },
    'hybrid': {
        'test_accuracy': float(hybrid_acc),
        'test_f1': float(hybrid_f1),
        'weights': hybrid_weights
    }
}

# Convert numpy to python types
for model_key in results:
    if 'history' in results[model_key]:
        for metric_key in results[model_key]['history']:
            results[model_key]['history'][metric_key] = [
                float(x) for x in results[model_key]['history'][metric_key]
            ]

with open('../artifacts/training_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"✓ Training results saved to artifacts/training_results.json")

## 11. Final Summary

In [ ]:
print("\n" + "="*70)
print("TRAINING COMPLETE - FINAL SUMMARY")
print("="*70)

print(f"\nBest Model: {'Hybrid' if hybrid_acc >= max(bert_test_acc, roberta_test_acc) else ('BERT' if bert_test_acc > roberta_test_acc else 'RoBERTa')}")
print(f"Best Test Accuracy: {max(bert_test_acc, roberta_test_acc, hybrid_acc):.4f}")

print(f"\nTest Set Results:")
print(f"  BERT - Accuracy: {bert_test_acc:.4f}, Macro F1: {bert_test_f1:.4f}")
print(f"  RoBERTa - Accuracy: {roberta_test_acc:.4f}, Macro F1: {roberta_test_f1:.4f}")
print(f"  Hybrid - Accuracy: {hybrid_acc:.4f}, Macro F1: {hybrid_f1:.4f}")

print(f"\nSaved Artifacts:")
print(f"  - artifacts/bert_model.pt")
print(f"  - artifacts/roberta_model.pt")
print(f"  - artifacts/training_results.json")

print(f"\nNext Steps:")
print(f"  1. Fine-tune hyperparameters (learning rate, batch size)")
print(f"  2. Add data augmentation techniques")
print(f"  3. Experiment with different ensemble weights")
print(f"  4. Create prediction API (see predict.py)")
print(f"  5. Deploy to production")

print(f"\n" + "="*70)